# Finance Module — Workflow Demo

This notebook demonstrates the two classes in the finance module:

1. **`ClimateVarEngine`** — Monte Carlo Climate Value-at-Risk. Propagates five layers of
   uncertainty (emission sampling, ERA5 wind bias, spatial extent, carbon price,
   regulatory pass-through) through 10,000 simulation paths.

2. **`TransitionRiskAnalyzer`** — Deterministic transition-risk scenarios. Maps the
   satellite methane estimate to carbon-cost exposure and Mild/Moderate/Severe
   bond + equity stress scenarios for PGE Polska Grupa Energetyczna.

**Prerequisites:** `numpy` (scipy optional, a shim is provided).
Run from the `methane-api/` repo root or from `scripts/finance/`.

In [ ]:
import sys
from pathlib import Path

# Ensure the scripts/ directory is on the path so imports work from any working directory
repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print(f'Repo root resolved to: {repo_root}')

In [ ]:
import numpy as np

from finance.finance_climate_var import (
    ClimateVarEngine,
    EmissionParams,
    UncertaintyParams,
    CarbonPriceParams,
)
from finance.finance_transition_risk import (
    TransitionRiskAnalyzer,
    BELCHATOW_EVIDENCE,
    PGE_PROFILE,
)

print('Imports OK')

## Part 1 — Monte Carlo Climate VaR

### 1a. Default run (Belchatow 30-record dataset, EUR 70/tCO2e central price)

In [ ]:
# Instantiate with default parameters
engine = ClimateVarEngine()

# Run 10,000-path simulation
results, liab_100, liab_20, q_annual = engine.Run(n_sim=10_000, seed=42)

# Print the standard summary table
engine.PrintSummary(results)

### 1b. Inspect the layer configuration

In [ ]:
import json
print('Uncertainty layers:')
for layer in results['metadata']['layers']:
    print(f"  {layer['layer']}: {layer['distribution']}")

### 1c. Plot the liability distribution (GWP100 vs GWP20)

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, liab, gwp_label, color in [
        (axes[0], liab_100, 'GWP100 (factor 28)', '#2166ac'),
        (axes[1], liab_20,  'GWP20  (factor 83)', '#d6604d'),
    ]:
        ax.hist(liab, bins=80, color=color, alpha=0.75, edgecolor='none')
        var99 = np.percentile(liab, 99)
        var95 = np.percentile(liab, 95)
        ax.axvline(np.mean(liab),  color='black',  lw=1.5, linestyle='--', label=f'Mean {np.mean(liab):.2f}M')
        ax.axvline(var95,          color='#fdae61', lw=1.5, linestyle='-',  label=f'VaR 95 {var95:.2f}M')
        ax.axvline(var99,          color='#d7191c', lw=1.5, linestyle='-',  label=f'VaR 99 {var99:.2f}M')
        ax.set_xlabel('Annual Liability (M EUR)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Climate VaR Distribution — {gwp_label}')
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('climate_var_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to climate_var_distribution.png')

except ImportError:
    print('matplotlib not installed — skipping plot.')
    print(f"GWP100: mean={np.mean(liab_100):.2f}M  VaR99={np.percentile(liab_100,99):.2f}M")
    print(f"GWP20:  mean={np.mean(liab_20):.2f}M   VaR99={np.percentile(liab_20,99):.2f}M")

### 1d. Customise parameters — what if carbon price rises to EUR 120/tCO2e?

In [ ]:
high_price_engine = ClimateVarEngine(
    carbon_price_params=CarbonPriceParams(central=120.0, log_vol=0.30)
)
hi_results, hi_100, hi_20, _ = high_price_engine.Run(n_sim=10_000, seed=42)

base_var99 = results['liability_gwp100']['var_99']
hi_var99   = hi_results['liability_gwp100']['var_99']

print(f"VaR 99 at EUR 70/tCO2e  : {base_var99:.2f} M EUR (GWP100)")
print(f"VaR 99 at EUR 120/tCO2e : {hi_var99:.2f} M EUR (GWP100)")
print(f"Uplift                  : {(hi_var99/base_var99 - 1):.0%}")

### 1e. Save results to JSON

In [ ]:
# Append tau sensitivity before saving
results['tau_sensitivity'] = engine.RunTauSensitivity(n_sim=5_000, seed=43)
engine.SaveResults(results)
print('Saved.')

---
## Part 2 — Deterministic Transition Risk Scenarios

### 2a. Default run

In [ ]:
analyzer = TransitionRiskAnalyzer()
payload  = analyzer.Run()
analyzer.PrintReport(payload)

### 2b. Carbon-cost table as a DataFrame

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(payload['carbon_cost_table'])
    df['exposure_eur_mean_m']  = df['exposure_eur_mean']  / 1e6
    df['exposure_eur_lower_m'] = df['exposure_eur_lower'] / 1e6
    df['exposure_eur_upper_m'] = df['exposure_eur_upper'] / 1e6
    display(df[['price_case', 'eur_per_tco2e',
                'exposure_eur_mean_m', 'exposure_eur_lower_m', 'exposure_eur_upper_m']]
            .rename(columns={
                'price_case': 'Scenario',
                'eur_per_tco2e': 'EUR/tCO2e',
                'exposure_eur_mean_m': 'Mean (M EUR)',
                'exposure_eur_lower_m': 'Lower 95% CI (M EUR)',
                'exposure_eur_upper_m': 'Upper 95% CI (M EUR)',
            }))
except ImportError:
    print('pandas not installed — printing raw table:')
    for r in payload['carbon_cost_table']:
        print(f"  {r['price_case']:10s} EUR {r['eur_per_tco2e']:3.0f}/tCO2e  "
              f"mean={r['exposure_eur_mean']/1e6:.2f}M EUR")

### 2c. Stress table

In [ ]:
print(f"CR01 (bond): EUR {payload['stress_table']['cr01_per_bp_eur']:,.0f} per bp\n")
print(f"{'Tier':<10} {'Equity P&L':>14} {'Bond P&L':>14} {'Combined':>14}")
print('-' * 55)
for r in payload['stress_table']['rows']:
    print(f"{r['tier']:<10} {r['equity_pnl_eur']/1e3:>+13.0f}K "
          f"{r['spread_pnl_eur']/1e3:>+13.0f}K "
          f"{r['combined_pnl_eur']/1e3:>+13.0f}K")

### 2d. Save results

In [ ]:
analyzer.SaveResults(payload)
print('Saved.')